# PHB case study

In [48]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ContiDesigner

ContiModel = ContiDesigner.ContiModel
ContiSolver = ContiDesigner.Solver
ContiPlotter = ContiDesigner.Plotter
DEFAULT_PROCESSES = ContiDesigner.DEFAULT_PROCESSES


In [49]:
# set up
params_base = DEFAULT_PROCESSES["PHB"]
params_base["growth_initial_state"] = [1, 0, 0.0]
initials = params_base["growth_initial_state"] * 5

params_base["stage2_mu_factor"] = 0.05

params_base["N_reactors"] = 5
params_base["growth_flags"] = [True, True, False, False, False]
params_base["Fs"] = [0.222, 0.2416, 0.2639, 0.2858, 0.3063]
params_base["Vs"] = [1.6, 1.6, 1.7, 1.7, 2.4]
params_base["pi0s"] = [0.0, 0, 0.23, 0.23, 0.23]
params_base["pi1s"] = [0.13, 0.13, 0.0, 0.0, 0.0]
params_base["sfs"] = [67, 500, 500, 500, 500]
model = ContiModel(params_base)
solver= ContiSolver(model)
plotter = ContiPlotter(model, solver)
print(model.mu_max,model.mu_stage2())


0.25 [0.0125, 0, 0, 0]


In [50]:
time_evol = plotter.plot_time_evolution()
time_evol.update_layout(
    title="Time Evolution in each reactor",
    title_x=0.5,
    legend=dict(orientation="h", y=-0.1, x=0.5, xanchor="center"),
)


In [51]:
from scipy.interpolate import interp1d
path = "./"

phb_df_original = pd.read_csv(path+'data-fig4a.csv', index_col=0)  # The CSV with columns: t, PHB_R1, PHB_R3
cdm_df_original = pd.read_csv(path+'data-fig4b.csv', index_col=0)  # The CSV with columns: t, CDM_R1, CDM_R3
phb_df_original.columns = ['PHB_R1', 'PHB_R2', 'PHB_R3', 'PHB_R4', 'PHB_R5']
cdm_df_original.columns = ['CDM_R1', 'CDM_R2', 'CDM_R3', 'CDM_R4', 'CDM_R5']

# %%

def align_dfs_time(phb_df, cdm_df):
    # Extract time columns as numpy arrays
    if len(phb_df) != len(cdm_df):
        raise ValueError("DataFrames must have the same number of rows to average times.")
    avg_times = (phb_df.index.to_numpy() + cdm_df.index.to_numpy()) / 2
    
    phb_aligned = phb_df.copy()
    phb_aligned.index = avg_times

    cdm_aligned = cdm_df.copy()
    cdm_aligned.index = avg_times

    return phb_aligned, cdm_aligned
phb_df, total_cdm_df = align_dfs_time(phb_df_original, cdm_df_original)


# %%

def extract_total_cdm(cdm_df, phb_df):
    # exclude the product (PHB) from total biomass (CDM) to get the actual biomass (X)
    total_bm = pd.DataFrame(index=cdm_df.index)
    for phb_col in phb_df.columns:
        cdm_col = phb_col.replace("PHB_", "CDM_")
        if cdm_col in cdm_df.columns:
            total_bm[cdm_col] = cdm_df[cdm_col] - phb_df[phb_col]
        else:
            # If column missing in PHB, just copy CDM as is
            print(f"Warning: {cdm_col} not found in CDM dataframe.")
    return total_bm


cdm_df = extract_total_cdm(total_cdm_df, phb_df)


# %%
fig = go.Figure()

fig.add_trace(go.Scatter(x=cdm_df.index, y=cdm_df['CDM_R1'], 
                         mode='markers+lines', name='Reactor 1 Data', 
                         marker=dict(color='blue', symbol='circle'), line=dict(dash='dot')))
fig.add_trace(go.Scatter(x=cdm_df.index, y=cdm_df['CDM_R2'], 
                         mode='markers+lines', name='Reactor 2 Data', 
                         marker=dict(color='red', symbol='x'), line=dict(dash='dot')))
fig.add_trace(go.Scatter(x=cdm_df.index, y=cdm_df['CDM_R3'], 
                         mode='markers+lines', name='Reactor 3 Data', 
                         marker=dict(color='green', symbol='triangle-up'), line=dict(dash='dot')))
fig.add_trace(go.Scatter(x=cdm_df.index, y=cdm_df['CDM_R4'], 
                         mode='markers+lines', name='Reactor 4 Data', 
                         marker=dict(color='magenta', symbol='x'), line=dict(dash='dot')))
fig.add_trace(go.Scatter(x=cdm_df.index, y=cdm_df['CDM_R5'], 
                         mode='markers+lines', name='Reactor 5 Data', 
                         marker=dict(color='cyan', symbol='triangle-up'), line=dict(dash='dot')))

fig.update_layout(
    title="CDM Concentration vs Time",
    xaxis_title="Time [h]",
    yaxis_title="CDM Concentration [g/L]",
    legend_title="Legend",
    width=800,
    height=600,
    template="simple_white",
)

fig.show()

# %%

fig = go.Figure()

fig.add_trace(go.Scatter(x=phb_df.index, y=phb_df['PHB_R1'], 
                         mode='markers+lines', name='Reactor 1 Data', 
                         marker=dict(color='blue', symbol='circle'), line=dict(dash='dot')))
fig.add_trace(go.Scatter(x=phb_df.index, y=phb_df['PHB_R2'], 
                         mode='markers+lines', name='Reactor 2 Data', 
                         marker=dict(color='red', symbol='x'), line=dict(dash='dot')))
fig.add_trace(go.Scatter(x=phb_df.index, y=phb_df['PHB_R3'], 
                         mode='markers+lines', name='Reactor 3 Data', 
                         marker=dict(color='green', symbol='triangle-up'), line=dict(dash='dot')))
fig.add_trace(go.Scatter(x=phb_df.index, y=phb_df['PHB_R4'], 
                         mode='markers+lines', name='Reactor 4 Data', 
                         marker=dict(color='magenta', symbol='x'), line=dict(dash='dot')))
fig.add_trace(go.Scatter(x=phb_df.index, y=phb_df['PHB_R5'], 
                         mode='markers+lines', name='Reactor 5 Data',
                         marker=dict(color='cyan', symbol='triangle-up'), line=dict(dash='dot')))

fig.update_layout(
    title="PHB Concentration vs Time",
    xaxis_title="Time [h]",
    yaxis_title="PHB Concentration [g/L]",
    legend_title="Legend",
    width=800,
    height=600,
    template="simple_white",
)

fig.show()
#
# %%

phb_ss = phb_df.iloc[-1]
cdm_ss = cdm_df.iloc[-1]

print(f"Steady states in reactor 1:\n"
      f"CDM = {cdm_ss['CDM_R1']:.2f} g/L, {phb_ss['PHB_R1']:.2f} g/L, \n"
      f"Steady states in reactor 2:\n"
      f"CDM = {cdm_ss['CDM_R2']:.2f} g/L, PHB = {phb_ss['PHB_R2']:.2f} g/L, \n"
      f"Steady states in reactor 3:\n"
      f"CDM = {cdm_ss['CDM_R3']:.2f} g/L, PHB = {phb_ss['PHB_R3']:.2f} g/L\n"
      f"Steady states in reactor 4:\n"
      f"CDM = {cdm_ss['CDM_R4']:.2f} g/L, PHB = {phb_ss['PHB_R4']:.2f} g/L\n"
      f"Steady states in reactor 5:\n"
      f"CDM = {cdm_ss['CDM_R5']:.2f} g/L, PHB = {phb_ss['PHB_R5']:.2f} g/L\n"
      )

Steady states in reactor 1:
CDM = 25.65 g/L, 0.19 g/L, 
Steady states in reactor 2:
CDM = 24.91 g/L, PHB = 16.13 g/L, 
Steady states in reactor 3:
CDM = 21.58 g/L, PHB = 38.50 g/L
Steady states in reactor 4:
CDM = 15.62 g/L, PHB = 54.35 g/L
Steady states in reactor 5:
CDM = 16.76 g/L, PHB = 62.61 g/L

